# 03 — Baselines Classiques : Correction Gamma et CLAHE
**Projet NightVision Gen — Références sans apprentissage**

Avant d'entraîner le DDPM, nous implémentons et évaluons deux méthodes classiques :
1. **Correction Gamma** : $I_{\text{out}} = I^{1/\gamma}$ ($\gamma \in [1.5, 2.2, 2.8]$).
2. **CLAHE (Contrast Limited Adaptive Histogram Equalization)** appliquée sur le canal de luminance L* de l'espace couleur LAB.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

sys.path.append(str(Path("..").resolve() / "src"))

from nightvision.evaluation.comparison import apply_gamma_correction, apply_clahe
from nightvision.evaluation.image_metrics import ImageMetricsEvaluator
from nightvision.data.dataset import SyntheticPairDataset

print("✅ Modules Baselines et Métriques importés.")

### 1. Comparaison Visuelle : Sombre vs Gamma vs CLAHE vs Référence

In [ ]:
ds = SyntheticPairDataset(length=1, image_size=256)
item = ds[0]
low = item["low_orig"].permute(1, 2, 0).numpy()
ref = item["normal_orig"].permute(1, 2, 0).numpy()

# Application des baselines
gamma_15 = apply_gamma_correction(low, gamma=1.5)
gamma_22 = apply_gamma_correction(low, gamma=2.2)
clahe_img = apply_clahe(low, clip_limit=3.0)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(low); axes[0].set_title("1. Sombre Original"); axes[0].axis("off")
axes[1].imshow(gamma_15); axes[1].set_title("2. Gamma (γ=1.5)"); axes[1].axis("off")
axes[2].imshow(gamma_22); axes[2].set_title("3. Gamma (γ=2.2)"); axes[2].axis("off")
axes[3].imshow(clahe_img); axes[3].set_title("4. CLAHE"); axes[3].axis("off")
axes[4].imshow(ref); axes[4].set_title("5. Référence Claire"); axes[4].axis("off")
plt.tight_layout()
plt.show()

### 2. Calcul des Métriques PSNR et SSIM

In [ ]:
evaluator = ImageMetricsEvaluator(device="cpu")

methods = {
    "Sombre Original": low,
    "Gamma 1.5": gamma_15,
    "Gamma 2.2": gamma_22,
    "CLAHE": clahe_img,
}

print(f"{'Méthode':<18} | {'PSNR (dB)':<10} | {'SSIM':<10}")
print("-" * 44)
for name, img in methods.items():
    metrics = evaluator.evaluate_pair(img, ref)
    print(f"{name:<18} | {metrics['psnr']:<10.2f} | {metrics['ssim']:<10.4f}")

### 3. Conclusion sur les Baselines
Les méthodes classiques augmentent la visibilité mais présentent des limites fortes : amplification du bruit dans les zones sombres et saturation des couleurs. Ces métriques serviront de point de comparaison direct pour les modèles DDPM A, B et C.